#  Calculation of features from available libraries

In [1]:
import sys
sys.path.insert(0, '/home/storage/fortimtb/CuadernoTrabajo/bopfoxfeaturizer/')

import os
import pandas as pd

dataset = 'Fe-Mo'
system=dataset.replace('-','')
from BopFoxFeaturizer.Featurizer import Featurizer

BS = pd.read_pickle(os.path.join(dataset, 'FullyCuratedParsedBriefSummary.pkl'))
AtomsObjects = pd.read_pickle(os.path.join(dataset, 'Atomsobjects', f'{system}-POSCAR-initial-rescaled-PymatgenStructures.pkl')).dropna()
BS.dropna(inplace=True)

## Prepare Extra features

In [2]:
Features = Featurizer(BS)

In [3]:
DatasetCompositionFeatures = Features.get_fractions_by_components()

In [4]:
DatasetMagneticFeature = Features.Mag
#DatasetMagneticFeature = DatasetMagneticFeature.str.replace('FM','1.')
#DatasetMagneticFeature = DatasetMagneticFeature.str.replace('NM','0.').astype(float)
DatasetMagneticFeature.name = 'Mag'

In [5]:
StructureNameFeature = BS.Phase

In [6]:
StructureNameFeature.name='Structure'

In [7]:
DatasetFeatures = pd.concat((DatasetCompositionFeatures, DatasetMagneticFeature, StructureNameFeature), axis=1)

In [8]:
from sklearn.preprocessing import  OneHotEncoder

In [9]:
encoder = OneHotEncoder()

In [10]:
encoder.fit(DatasetFeatures[['Structure', 'Mag']])

OneHotEncoder()

In [11]:
transformed = encoder.transform(DatasetFeatures[['Structure', 'Mag']])

In [12]:
dftransformed = pd.DataFrame(transformed.toarray(), columns = encoder.get_feature_names_out(), index = DatasetFeatures.index)

In [13]:
DatasetFeatures.drop(columns = ['Mag', 'Structure'],  inplace=True)

In [14]:
DatasetFeatures = pd.concat([DatasetFeatures, dftransformed], axis = 1)

In [15]:
datasetfeatureslocation = os.path.join(dataset, 'Descriptors','DatasetFeatures.pkl')

# Matminer Features 

In [16]:
from Tools.DatasetTools.GetPymatgenFeatures import *

In [17]:
descriptorslocation = os.path.join(dataset, 'Descriptors')
mmflatomic = os.path.join(descriptorslocation, 'matminer_atomic_features.pkl')
mmfdensity = os.path.join (descriptorslocation, 'matminer_density_features.pkl')
mmfcomposition =  os.path.join (descriptorslocation,'matminer_composition_features.pkl')
mmfstructure =  os.path.join (descriptorslocation,'matminer_structure_features.pkl')
mmsoapfeatures = os.path.join(descriptorslocation, 'matminer_soap_features.pkl')


BS['chemical_formula'] = get_chemical_formula(BS)

In [18]:
BS['composition'] = StrToComposition().featurize_dataframe(BS, "chemical_formula")['composition']

StrToComposition:   0%|          | 0/289 [00:00<?, ?it/s]

In [19]:
BS['atoms_objects'] = AtomsObjects

In [20]:
AtomicFeaturesMagpie = load_features(mmflatomic, BS, which='atomic')
DensitiFeatures= load_features(mmfdensity, BS, which='density')
CompositionFeatures = load_features(mmfcomposition, BS, which='composition')
# SOAPFeatures = load_features(mmfstructure, BS, which='soap')
# SOAP doesnt work from matminer
# StructureFeatures = load_features(mmfstructure, BS, which='structure')

In [21]:
AtomicFeaturesMagpie.columns = AtomicFeaturesMagpie.columns.str.replace('MagpieData ','')
AtomicFeaturesMagpie.dropna(axis=1, inplace = True)
AtomicFeaturesMagpie.describe()

,minimum Number,maximum Number,range Number,mean Number,avg_dev Number,mode Number,minimum MendeleevNumber,maximum MendeleevNumber,range MendeleevNumber,mean MendeleevNumber,...,range GSmagmom,mean GSmagmom,avg_dev GSmagmom,mode GSmagmom,minimum SpaceGroupNumber,maximum SpaceGroupNumber,range SpaceGroupNumber,mean SpaceGroupNumber,avg_dev SpaceGroupNumber,mode SpaceGroupNumber
count,289.000000,289.000000,289.000000,289.000000,289.000000,289.000000,289.000000,289.000000,289.000000,289.000000,...,289.000000,289.000000,289.000000,289.000000,289.0,289.0,289.0,289.0,289.0,289.0
mean,27.217993,40.948097,13.730104,39.126264,2.732522,40.615917,50.328720,54.619377,4.290657,50.898042,...,1.811226,0.379093,0.360465,0.182583,229.0,229.0,0.0,229.0,0.0,229.0
std,4.250517,3.972222,5.592327,3.989525,2.268611,4.505539,1.241319,1.328287,1.747602,1.246727,...,0.737720,0.526284,0.299267,0.594355,0.0,0.0,0.0,0.0,0.0,0.0
min,26.000000,26.000000,0.000000,26.000000,0.000000,26.000000,50.000000,50.000000,0.000000,50.000000,...,0.000000,0.000000,0.000000,0.000000,229.0,229.0,0.0,229.0,0.0,229.0
25%,26.000000,42.000000,16.000000,39.061224,0.904892,42.000000,50.000000,55.000000,5.000000,50.212766,...,2.110663,0.089815,0.119370,0.000000,229.0,229.0,0.0,229.0,0.0,229.0
50%,26.000000,42.000000,16.000000,40.545455,2.223930,42.000000,50.000000,55.000000,5.000000,50.454545,...,2.110663,0.191878,0.293373,0.000000,229.0,229.0,0.0,229.0,0.0,229.0
75%,26.000000,42.000000,16.000000,41.319149,4.444444,42.000000,50.000000,55.000000,5.000000,50.918367,...,2.110663,0.387673,0.586295,0.000000,229.0,229.0,0.0,229.0,0.0,229.0
max,42.000000,42.000000,16.000000,42.000000,7.933884,42.000000,55.000000,55.000000,5.000000,55.000000,...,2.110663,2.110663,1.046610,2.110663,229.0,229.0,0.0,229.0,0.0,229.0


In [22]:
DensitiFeatures.dropna(axis=1, inplace=True)
if DensitiFeatures.shape[1] > 0:
    DensitiFeatures.describe()

In [23]:
CompositionFeatures.dropna(axis=1, inplace=True)
CompositionFeatures.describe()

,HOMO_energy,LUMO_energy,gap_AO,band center,max ionic char,avg ionic char,0-norm,2-norm,3-norm,5-norm,7-norm,10-norm
count,289.000000,289.000000,289.0,289.000000,289.000000,289.000000,289.000000,289.000000,289.000000,289.000000,289.000000,289.000000
mean,-0.162972,-0.162972,0.0,3.939285,0.023047,0.002293,1.858131,0.907015,0.896017,0.892222,0.891545,0.891299
std,0.035292,0.035292,0.0,0.028412,0.009387,0.001904,0.349520,0.081019,0.097394,0.105097,0.106785,0.107461
min,-0.295049,-0.295049,0.0,3.918947,0.000000,0.000000,1.000000,0.710023,0.635124,0.583580,0.564976,0.553682
25%,-0.153347,-0.153347,0.0,3.923732,0.026858,0.000759,2.000000,0.849837,0.835550,0.833387,0.833335,0.833333
50%,-0.153347,-0.153347,0.0,3.929177,0.026858,0.001867,2.000000,0.927903,0.925021,0.924856,0.924855,0.924855
75%,-0.153347,-0.153347,0.0,3.939644,0.026858,0.003730,2.000000,0.971311,0.970883,0.970874,0.970874,0.970874
max,-0.153347,-0.153347,0.0,4.032960,0.026858,0.006659,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000


# Pyscal features 

In [24]:
atomsobjectlocation = os.path.join(dataset, 'Atomsobjects')
atomsobjectfile = os.path.join(atomsobjectlocation,f'{system}-POSCAR-initial-rescaled-AtomsObjects.pkl')

In [25]:
from tqdm.auto import tqdm
from Tools.DatasetTools import pyscalfeaturizers as pf
from BopFoxFeaturizer.struct_db import struct_db
AtomsObjects = pd.read_pickle(atomsobjectfile).dropna()

In [26]:
AtomsObjects

,atoms,file
index,,
Fe_pv8Mo_sv22.sigma-BBABB.FM,"(Atom('Fe', [4.144207147225807, 1.172551050294...",[Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/sigma-BBABB....
Fe_pv10Mo_sv20.sigma-ABBAB.FM,"(Atom('Fe', [0.0, 0.0, 0.0], index=0), Atom('F...",[Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/sigma-ABBAB....
Fe_pv4Mo_sv20.C36-ABBBB.FM,"(Atom('Fe', [0.0, 0.0, 1.4500024886822753], in...",[Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/C36-ABBBB.FM...
Fe_pv3Mo_sv10.mu-ABBBA.FM,"(Atom('Fe', [0.0, 0.0, 0.0], index=0), Atom('F...",[Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/mu-ABBBA.FM/...
Fe_pv5Mo_sv24.chi-AABB.FM,"(Atom('Fe', [0.0, 0.0, 0.0], index=0), Atom('F...",[Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/chi-AABB.FM/...
...,...,...
Fe_pv3Mo_sv10.mu-ABBBA.NM,"(Atom('Fe', [0.0, 0.0, 0.0], index=0), Atom('F...",[Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/mu-ABBBA/rel...
Fe_pv8Mo_sv22.sigma-BBABB.NM,"(Atom('Fe', [4.144207147225807, 1.172551050294...",[Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/sigma-BBABB/...
Fe_pv1Mo_sv3.L12-AB3.FM,"(Atom('Fe', [0.0, 0.0, 0.0], index=0), Atom('M...",[Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/L12-AB3.FM/r...


##  Coordination Numbers

In [27]:
Sorters = pd.read_pickle('Fe-Mo/Atomsobjects/SORTERS.pkl')

In [28]:
Sorters.index = Sorters.index.str.strip()

In [29]:
relevant_sorters = Sorters[AtomsObjects.file.map(lambda v: v[0])]

In [30]:
SublatticeTags = pd.read_pickle('Fe-Mo/Atomsobjects/SUBLATICETAGS.pkl')
SublatticeTags.index = SublatticeTags.index.str.strip()

In [31]:
relevantSublatticeTags = SublatticeTags[relevant_sorters.index]

In [32]:
relevantSublatticeTags

Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/sigma-BBABB.FM/relax/xc=PBE-PAW.E=450.dk=0.020/POSCAR.initial    [A, A, B, B, B, B, C, C, C, C, C, C, C, C, D, ...
Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/sigma-ABBAB.FM/relax/xc=PBE-PAW.E=450.dk=0.020/POSCAR.initial    [A, A, B, B, B, B, C, C, C, C, C, C, C, C, D, ...
Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/C36-ABBBB.FM/relax/xc=PBE-PAW.E=450.dk=0.020/POSCAR.initial      [A, A, A, A, B, B, B, B, C, C, C, C, D, D, D, ...
Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/mu-ABBBA.FM/relax/xc=PBE-PAW.E=450.dk=0.020/POSCAR.initial                 [A, B, B, B, B, B, B, C, C, D, D, E, E]
Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/chi-AABB.FM/relax/xc=PBE-PAW.E=450.dk=0.020/POSCAR.initial       [A, B, B, B, B, C, C, C, C, C, C, C, C, C, C, ...
                                                                                                                        ...                        
Fe-Mo/Fe_pv-Mo_sv/POSCAR-initial/mu-ABBBA/relax/xc=PBE-PAW.E=450.dk=0.020/POSCAR.initial                    [A, 

replace index of relevant data by sample

In [33]:
import numpy as np

In [34]:
def get_min_ifnotempty(thelist):
    if len(thelist) > 0:
        return np.min(thelist)

In [35]:
type(relevant_sorters[0])

list

In [36]:
import pdb

In [37]:
def substract_reference_if_notnan(themin,thelist):
    try:
        return np.array(thelist)-themin
    except:
        pdb.set_trace()
        pass

In [38]:
references = relevant_sorters.map(get_min_ifnotempty)

goodsorters = {}
for index, arrayfile in AtomsObjects.file.iteritems():
    file = arrayfile[0]
    goodsorters[index]=substract_reference_if_notnan(references[file], relevant_sorters[file])

In [40]:
featurizers = [pf.pyscal_steinhardt, pf.pyscal_cn] #, get_steinhardt]
pyscal_features = [feature.__name__ for feature in featurizers]

pyscalsteinhardt = os.path.join(descriptorslocation, 'pyscal_steinhardt.kpl')

if os.path.exists(pyscalsteinhardt):
    PyscalFeatures = pd.read_pickle(pyscalsteinhardt)
else:
    PyscalFeatures = pf.featurize_many(AtomsObjects,  featurizers, colid='atoms')
    expanded_ste = pf.expand_features(PyscalFeatures.pyscal_steinhardt, 'pyscal_steinhardt')
    PyscalFeatures = pd.concat([expanded_ste, PyscalFeatures.pyscal_cn], axis=1)
    PyscalFeatures.to_pickle(pyscalsteinhardt)

pyscal_steinhardt


  0%|          | 0/289 [00:00<?, ?it/s]

pyscal_cn


  0%|          | 0/289 [00:00<?, ?it/s]

In [ ]:
PyscalFeatures

# Dataset features from CN 

The first feature that we would like to have is the count of each CP in each sample. for that we construct a vector in the following way:

$$ N_{CN}^i = \#^i CN $$

## amount of sites in structure

In [ ]:
import Tools.DatasetTools.GeneralFeaturizer as gf

In [ ]:
from importlib.machinery import SourceFileLoader
gf = SourceFileLoader('gf','Tools/DatasetTools/GeneralFeaturizer.py').load_module()

In [ ]:
CN = gf.featurize_series(PyscalFeatures['pyscal_cn'], PyscalFeatures.pyscal_cn, normalization='NCP', return0 = False)
newcolumns = ['N'+col for col in CN.columns]
CN.columns = newcolumns

In [ ]:
CN

## Composition on CP

Next feature we want is the composition in each CP. for this we choose to represent the elment numerically by their atomic numbers, and the CP-resolved composition becomes the average atomc numbers,

$$ Z_{CP} ^i = \dfrac{1}{n_{at}^i} \sum_{at \in CP} Z_{at} $$

In [ ]:
from mendeleev import element

In [ ]:
AtomicNumbers=AtomsObjects.atoms.map(lambda a: a.numbers)
AtomicNumbers.name = 'AtomicNumbers'

In [ ]:
symbols = dataset.split('-')

In [ ]:
volums = {symb: element(symb).atomic_volume for symb in symbols}

In [ ]:
AtomicVolumes = AtomsObjects.atoms.map(lambda a: [volums[at] for at in a.get_chemical_symbols()])

In [ ]:
CPVol = gf.featurize_series(AtomicVolumes, PyscalFeatures.pyscal_cn, return0=False)
newcolumns = ['V'+col for col in CPVol.columns]
CPVol.columns =  newcolumns

In [ ]:
CPComp = gf.featurize_series(AtomicNumbers, PyscalFeatures.pyscal_cn, return0=False)
newcolumns = ['Z'+col for col in CPComp.columns]
CPComp.columns = newcolumns

In [ ]:
DatasetFeatures = pd.concat([DatasetFeatures, CN, CPComp,CPVol], axis=1)

In [ ]:
DatasetFeatures.to_pickle(datasetfeatureslocation)

In [ ]:
DatasetFeatures

## Stainhardt parameters 

From the Steinhardt parameters obtained by Pyscal library, we also want to average over the coordination polyhedra. This time we are also saving the total average for each parameter.

$$ q_{j, CP} ^i = \dfrac{1}{n_{at}^i}\sum _{at \in CP} q_{j, at} ^i $$

In [ ]:
thisFeatures = PyscalFeatures[['pyscal_steinhardt_0','pyscal_steinhardt_1']]

In [ ]:
CNPyscal  = gf.featurize_many(thisFeatures, PyscalFeatures.pyscal_cn, [gf.cn_average])

In [ ]:
CNPyscal

In [ ]:
PyscalFeaturesFile = os.path.join(descriptorslocation,'CNAVPyscal.pkl')

In [ ]:
CNPyscal.to_pickle(PyscalFeaturesFile)

# Characterization of Dataset Features 

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns

In [ ]:
sns.pairplot(CN)

In [ ]:
sns.pairplot(CNPyscal)

# compare to alesya's descriptors

In [ ]:
previous = pd.read_csv('Fe-Mo/DATAforML_without_BOP.csv')

In [ ]:
previous = previous.set_index('index1')

In [ ]:
previous

In [ ]:
previous.columns

In [ ]:
DatasetFeatures.columns

### magnetism

In [ ]:
previous.Magnetism

In [ ]:
interception = previous.index.intersection(DatasetFeatures.index)

In [ ]:
interception

In [ ]:
(DatasetFeatures.Mag_FM.loc[interception] == previous.Magnetism.loc[interception]).sum()

### Z_CN

In [ ]:
previous.filter(regex='Z')

In [ ]:
DatasetFeatures.filter(regex='Z')

In [ ]:
previous.columns